In [1]:
from google.colab import drive
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np

In [2]:
# 1. Cargar el dataset desde tu archivo CSV
# 1. Carga de datos desde Drive
drive.mount('/content/drive')
dataset = load_dataset('csv', data_files='/content/drive/MyDrive/data/dataset_arroz_finetunig.csv')
# Dividir en entrenamiento (80%) y validación (20%)
dataset = dataset['train'].train_test_split(test_size=0.2)

Mounted at /content/drive


Generating train split: 0 examples [00:00, ? examples/s]

In [3]:
# 2. Cargar el Tokenizer (convierte texto a números para el modelo)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["texto_original"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

In [4]:
# 3. Cargar el modelo base
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# 4. Configurar el entrenamiento
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/resultado_entrenamiento",
    eval_strategy="epoch",
    per_device_train_batch_size=8,
    num_train_epochs=3, # Pocas épocas para evitar sobreajuste
    save_strategy="epoch"
)

In [6]:
# 5. Entrenar
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

In [7]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.700297
2,No log,0.699425
3,No log,0.700914


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=0.6979355812072754, metrics={'train_runtime': 130.9546, 'train_samples_per_second': 0.183, 'train_steps_per_second': 0.023, 'total_flos': 3179217567744.0, 'train_loss': 0.6979355812072754, 'epoch': 3.0})